In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# path constants
DATA = '../data/individual/processed'
PSY = '../data/individual/psychometric'

# HRV metric calculations
def calculate_hrv_metrics(ibi_data):
    valid_ibi = pd.Series(ibi_data)
    diff_nn_intervals = np.diff(valid_ibi)
    squared_diffs = np.square(diff_nn_intervals)
    rmssd = np.sqrt(np.mean(squared_diffs))
    sdnn = np.std(valid_ibi, ddof=1)
    return rmssd, sdnn

# filter IBI by questions
def get_ibi_data(questions, ibi_data):
    ibi_copy = ibi_data.copy()
    ibi_copy['datetime'] = pd.to_datetime(
        ibi_copy['datetime'], utc=True, errors='coerce'
    ).dt.tz_convert(None)
    ibi_filtered = ibi_copy[
        (ibi_copy['datetime'] >= questions['Question Start Time'].min()) &
        (ibi_copy['datetime'] <= questions['Question Answer Time'].max())
    ]
    return ibi_filtered['ibi'].values

# check anxiety thresholds
def determine_anxiety(rmssd, sdnn, rmssd_baseline, sdnn_baseline):
    general_anxiety = rmssd < rmssd_threshold or sdnn < sdnn_threshold
    individual_anxiety = rmssd < rmssd_baseline or sdnn < sdnn_baseline
    return general_anxiety, individual_anxiety

# process question type
def process_questions(question_type, psychometric_data, ibi_data,
                      rmssd_baseline, sdnn_baseline):
    questions_01 = psychometric_data[0][psychometric_data[0]['Type'] == question_type].copy()
    questions_02 = psychometric_data[1][psychometric_data[1]['Type'] == question_type].copy()
    questions_03 = psychometric_data[2][psychometric_data[2]['Type'] == question_type].copy()
    ibi_01 = get_ibi_data(questions_01, ibi_data[0])
    ibi_02 = get_ibi_data(questions_02, ibi_data[1])
    ibi_03 = get_ibi_data(questions_03, ibi_data[2])
    ibi_01 = ibi_01[ibi_01 > 0]
    ibi_02 = ibi_02[ibi_02 > 0]
    ibi_03 = ibi_03[ibi_03 > 0]
    rmssd_01, sdnn_01 = calculate_hrv_metrics(ibi_01)
    rmssd_02, sdnn_02 = calculate_hrv_metrics(ibi_02)
    rmssd_03, sdnn_03 = calculate_hrv_metrics(ibi_03)
    anxiety_general_01, anxiety_individual_01 = determine_anxiety(rmssd_01, sdnn_01, rmssd_baseline, sdnn_baseline)
    anxiety_general_02, anxiety_individual_02 = determine_anxiety(rmssd_02, sdnn_02, rmssd_baseline, sdnn_baseline)
    anxiety_general_03, anxiety_individual_03 = determine_anxiety(rmssd_03, sdnn_03, rmssd_baseline, sdnn_baseline)
    results = pd.DataFrame({
        'Test': ['Test 01', 'Test 02', 'Test 03'],
        'Start Time': [
            questions_01['Question Start Time'].min().strftime('%H:%M:%S'),
            questions_02['Question Start Time'].min().strftime('%H:%M:%S'),
            questions_03['Question Start Time'].min().strftime('%H:%M:%S')
        ],
        'End Time': [
            questions_01['Question Answer Time'].max().strftime('%H:%M:%S'),
            questions_02['Question Answer Time'].max().strftime('%H:%M:%S'),
            questions_03['Question Answer Time'].max().strftime('%H:%M:%S')
        ],
        'RMSSD': [rmssd_01, rmssd_02, rmssd_03],
        'SDNN': [sdnn_01, sdnn_02, sdnn_03],
        'General Anxiety (RMSSD)': ['Yes' if anxiety_general_01 else 'No', 'Yes' if anxiety_general_02 else 'No', 'Yes' if anxiety_general_03 else 'No'],
        'General Anxiety (SDNN)': ['Yes' if sdnn_01 < sdnn_threshold else 'No', 'Yes' if sdnn_02 < sdnn_threshold else 'No', 'Yes' if sdnn_03 < sdnn_threshold else 'No'],
        'Individual Anxiety (RMSSD)': ['Yes' if anxiety_individual_01 else 'No', 'Yes' if anxiety_individual_02 else 'No', 'Yes' if anxiety_individual_03 else 'No'],
        'Individual Anxiety (SDNN)': ['Yes' if sdnn_01 < sdnn_baseline else 'No', 'Yes' if sdnn_02 < sdnn_baseline else 'No', 'Yes' if sdnn_03 < sdnn_baseline else 'No']
    })
    return results

# anxiety thresholds
rmssd_threshold = 20
sdnn_threshold = 50

# load HR data
hr_baseline = pd.read_csv(f'{DATA}/hr.csv')

# load IBI data
ibi_baseline = pd.read_csv(f'{DATA}/ibi.csv')
ibi_01 = pd.read_csv(f'{DATA}/ibi_01.csv')
ibi_02 = pd.read_csv(f'{DATA}/ibi_02.csv')
ibi_03 = pd.read_csv(f'{DATA}/ibi_03.csv')

# validate baseline IBI
ibi_baseline = ibi_baseline[ibi_baseline['ibi'] > 0]

# baseline HRV metrics
rmssd_baseline, sdnn_baseline = calculate_hrv_metrics(ibi_baseline['ibi'])

# load psychometric data
psychometric_01 = pd.read_csv(f'{PSY}/Psychometric_Test_Results_01.csv')
psychometric_02 = pd.read_csv(f'{PSY}/Psychometric_Test_Results_02.csv')
psychometric_03 = pd.read_csv(f'{PSY}/Psychometric_Test_Results_03.csv')

# convert datetime columns
for df in [psychometric_01, psychometric_02, psychometric_03]:
    df['Question Start Time'] = pd.to_datetime(
        df['Question Start Time'], utc=True, errors='coerce'
    ).dt.tz_convert(None)
    df['Question Answer Time'] = pd.to_datetime(
        df['Question Answer Time'], utc=True, errors='coerce'
    ).dt.tz_convert(None)

# drop missing timestamps
psychometric_01 = psychometric_01.dropna(subset=['Question Start Time'])
psychometric_02 = psychometric_02.dropna(subset=['Question Start Time'])
psychometric_03 = psychometric_03.dropna(subset=['Question Start Time'])

# filter HADS questions
questions_hads_01 = psychometric_01[psychometric_01['Type'] == 'HADS'].copy()
questions_hads_02 = psychometric_02[psychometric_02['Type'] == 'HADS'].copy()
questions_hads_03 = psychometric_03[psychometric_03['Type'] == 'HADS'].copy()

# filter STAI-S questions
questions_stais_01 = psychometric_01[psychometric_01['Type'] == 'STAI-S'].copy()
questions_stais_02 = psychometric_02[psychometric_02['Type'] == 'STAI-S'].copy()
questions_stais_03 = psychometric_03[psychometric_03['Type'] == 'STAI-S'].copy()

# filter STAI-T questions
questions_stait_01 = psychometric_01[psychometric_01['Type'] == 'STAI-T'].copy()
questions_stait_02 = psychometric_02[psychometric_02['Type'] == 'STAI-T'].copy()
questions_stait_03 = psychometric_03[psychometric_03['Type'] == 'STAI-T'].copy()

# filter BFI questions
questions_bfi_01 = psychometric_01[psychometric_01['Type'] == 'BFI'].copy()
questions_bfi_02 = psychometric_02[psychometric_02['Type'] == 'BFI'].copy()
questions_bfi_03 = psychometric_03[psychometric_03['Type'] == 'BFI'].copy()

# filter FQ questions
questions_fq_01 = psychometric_01[psychometric_01['Type'] == 'FQ'].copy()
questions_fq_02 = psychometric_02[psychometric_02['Type'] == 'FQ'].copy()
questions_fq_03 = psychometric_03[psychometric_03['Type'] == 'FQ'].copy()

# batch processing lists
psychometric_data = [psychometric_01, psychometric_02, psychometric_03]
ibi_data = [ibi_01, ibi_02, ibi_03]
question_types = ['HADS', 'STAI-S', 'STAI-T', 'BFI', 'FQ']

print("Setup complete.")

In [ ]:
# baseline ibi hrv
if 'ibi' in ibi_baseline.columns:
    valid_ibi = ibi_baseline[(ibi_baseline['ibi'] > 300) & (ibi_baseline['ibi'] < 2000)]['ibi']

    diff_nn_intervals = np.diff(valid_ibi)
    squared_diffs = np.square(diff_nn_intervals)
    rmssd = np.sqrt(np.mean(squared_diffs))
    sdnn = np.std(valid_ibi, ddof=1)

    print(f"Baseline RMSSD: {rmssd:.2f} ms")
    print(f"Baseline SDNN: {sdnn:.2f} ms")
else:
    print("IBI data column not found. Please ensure the HR data includes 'ibi' measurements.")

In [ ]:
# Calculate HRV metrics per test
rmssd_bl, sdnn_bl = calculate_hrv_metrics(ibi_baseline['ibi'])
rmssd_01, sdnn_01 = calculate_hrv_metrics(ibi_01[ibi_01['ibi'] > 0]['ibi'])
rmssd_02, sdnn_02 = calculate_hrv_metrics(ibi_02[ibi_02['ibi'] > 0]['ibi'])
rmssd_03, sdnn_03 = calculate_hrv_metrics(ibi_03[ibi_03['ibi'] > 0]['ibi'])

# Display the results
print(f"Baseline RMSSD: {rmssd_bl:.2f} ms, SDNN: {sdnn_bl:.2f} ms")
print(f"Test 01 RMSSD: {rmssd_01:.2f} ms, SDNN: {sdnn_01:.2f} ms")
print(f"Test 02 RMSSD: {rmssd_02:.2f} ms, SDNN: {sdnn_02:.2f} ms")
print(f"Test 03 RMSSD: {rmssd_03:.2f} ms, SDNN: {sdnn_03:.2f} ms")

In [ ]:
# Calculate HRV metrics per test (alternate computation)
rmssd_bl, sdnn_bl = calculate_hrv_metrics(ibi_baseline['ibi'])
rmssd_01, sdnn_01 = calculate_hrv_metrics(ibi_01[ibi_01['ibi'] > 0]['ibi'])
rmssd_02, sdnn_02 = calculate_hrv_metrics(ibi_02[ibi_02['ibi'] > 0]['ibi'])
rmssd_03, sdnn_03 = calculate_hrv_metrics(ibi_03[ibi_03['ibi'] > 0]['ibi'])

print(f"Baseline RMSSD: {rmssd_bl:.2f} ms, SDNN: {sdnn_bl:.2f} ms")
print(f"Test 01 RMSSD: {rmssd_01:.2f} ms, SDNN: {sdnn_01:.2f} ms")
print(f"Test 02 RMSSD: {rmssd_02:.2f} ms, SDNN: {sdnn_02:.2f} ms")
print(f"Test 03 RMSSD: {rmssd_03:.2f} ms, SDNN: {sdnn_03:.2f} ms")

In [ ]:
# Calculate HRV metrics per test
rmssd_bl, sdnn_bl = calculate_hrv_metrics(ibi_baseline['ibi'])
rmssd_01, sdnn_01 = calculate_hrv_metrics(ibi_01[ibi_01['ibi'] > 0]['ibi'])
rmssd_02, sdnn_02 = calculate_hrv_metrics(ibi_02[ibi_02['ibi'] > 0]['ibi'])
rmssd_03, sdnn_03 = calculate_hrv_metrics(ibi_03[ibi_03['ibi'] > 0]['ibi'])

# Check anxiety per test
anxiety_general_01, anxiety_individual_01 = determine_anxiety(rmssd_01, sdnn_01, rmssd_bl, sdnn_bl)
anxiety_general_02, anxiety_individual_02 = determine_anxiety(rmssd_02, sdnn_02, rmssd_bl, sdnn_bl)
anxiety_general_03, anxiety_individual_03 = determine_anxiety(rmssd_03, sdnn_03, rmssd_bl, sdnn_bl)

# Display the results
print(f"Baseline RMSSD: {rmssd_bl:.2f} ms, SDNN: {sdnn_bl:.2f} ms")
print(f"Test 01 RMSSD: {rmssd_01:.2f} ms, SDNN: {sdnn_01:.2f} ms, General Anxiety: {'Yes' if anxiety_general_01 else 'No'}, Individual Anxiety: {'Yes' if anxiety_individual_01 else 'No'}")
print(f"Test 02 RMSSD: {rmssd_02:.2f} ms, SDNN: {sdnn_02:.2f} ms, General Anxiety: {'Yes' if anxiety_general_02 else 'No'}, Individual Anxiety: {'Yes' if anxiety_individual_02 else 'No'}")
print(f"Test 03 RMSSD: {rmssd_03:.2f} ms, SDNN: {sdnn_03:.2f} ms, General Anxiety: {'Yes' if anxiety_general_03 else 'No'}, Individual Anxiety: {'Yes' if anxiety_individual_03 else 'No'}")

In [ ]:
# Calculate HRV metrics per test
rmssd_bl, sdnn_bl = calculate_hrv_metrics(ibi_baseline['ibi'])
rmssd_01, sdnn_01 = calculate_hrv_metrics(ibi_01[ibi_01['ibi'] > 0]['ibi'])
rmssd_02, sdnn_02 = calculate_hrv_metrics(ibi_02[ibi_02['ibi'] > 0]['ibi'])
rmssd_03, sdnn_03 = calculate_hrv_metrics(ibi_03[ibi_03['ibi'] > 0]['ibi'])

# Check anxiety per test
anxiety_general_01, anxiety_individual_01 = determine_anxiety(rmssd_01, sdnn_01, rmssd_bl, sdnn_bl)
anxiety_general_02, anxiety_individual_02 = determine_anxiety(rmssd_02, sdnn_02, rmssd_bl, sdnn_bl)
anxiety_general_03, anxiety_individual_03 = determine_anxiety(rmssd_03, sdnn_03, rmssd_bl, sdnn_bl)

# Collect data for visualization
data = {
    'Baseline': (rmssd_bl, sdnn_bl, False, False),
    'Test 01': (rmssd_01, sdnn_01, anxiety_general_01, anxiety_individual_01),
    'Test 02': (rmssd_02, sdnn_02, anxiety_general_02, anxiety_individual_02),
    'Test 03': (rmssd_03, sdnn_03, anxiety_general_03, anxiety_individual_03)
}

# Create table
df = pd.DataFrame(data, index=['RMSSD', 'SDNN', 'General Anxiety', 'Individual Anxiety'])

# Plot RMSSD and SDNN
fig, ax = plt.subplots(2, 1, figsize=(10, 10))

# RMSSD Plot
ax[0].bar(df.columns, df.loc['RMSSD'], color=['green' if not anxiety else 'red' for anxiety in df.loc['General Anxiety']], alpha=0.7)
ax[0].axhline(rmssd_threshold, color='red', linestyle='--', label='RMSSD Threshold')
ax[0].set_title('RMSSD Across Tests')
ax[0].set_ylabel('RMSSD (ms)')
ax[0].legend()

# SDNN Plot
ax[1].bar(df.columns, df.loc['SDNN'], color=['green' if not anxiety else 'red' for anxiety in df.loc['General Anxiety']], alpha=0.7)
ax[1].axhline(sdnn_threshold, color='red', linestyle='--', label='SDNN Threshold')
ax[1].set_title('SDNN Across Tests')
ax[1].set_ylabel('SDNN (ms)')
ax[1].legend()

plt.tight_layout()
plt.show()
plt.close()

In [ ]:
# Calculate HRV metrics during HADS questions
ibi_hads_01 = get_ibi_data(questions_hads_01, ibi_01)
ibi_hads_02 = get_ibi_data(questions_hads_02, ibi_02)
ibi_hads_03 = get_ibi_data(questions_hads_03, ibi_03)

rmssd_hads_01, sdnn_hads_01 = calculate_hrv_metrics(ibi_hads_01)
rmssd_hads_02, sdnn_hads_02 = calculate_hrv_metrics(ibi_hads_02)
rmssd_hads_03, sdnn_hads_03 = calculate_hrv_metrics(ibi_hads_03)

# Check anxiety per test
anxiety_general_01, anxiety_individual_01 = determine_anxiety(rmssd_hads_01, sdnn_hads_01, rmssd_baseline, sdnn_baseline)
anxiety_general_02, anxiety_individual_02 = determine_anxiety(rmssd_hads_02, sdnn_hads_02, rmssd_baseline, sdnn_baseline)
anxiety_general_03, anxiety_individual_03 = determine_anxiety(rmssd_hads_03, sdnn_hads_03, rmssd_baseline, sdnn_baseline)

# Create results table
results_hads = pd.DataFrame({
    'Test': ['Test 01', 'Test 02', 'Test 03'],
    'RMSSD': [rmssd_hads_01, rmssd_hads_02, rmssd_hads_03],
    'SDNN': [sdnn_hads_01, sdnn_hads_02, sdnn_hads_03],
    'General Anxiety': ['Yes' if anxiety_general_01 else 'No', 'Yes' if anxiety_general_02 else 'No', 'Yes' if anxiety_general_03 else 'No'],
    'Individual Anxiety': ['Yes' if anxiety_individual_01 else 'No', 'Yes' if anxiety_individual_02 else 'No', 'Yes' if anxiety_individual_03 else 'No']
})

# Display results
print(f"Baseline RMSSD: {rmssd_baseline:.2f} ms, SDNN: {sdnn_baseline:.2f} ms\n")

for index, row in results_hads.iterrows():
    print(f"{row['Test']}:")
    print(f"  RMSSD during HADS: {row['RMSSD']:.2f} ms")
    print(f"  SDNN during HADS: {row['SDNN']:.2f} ms")
    print(f"  General Anxiety: {row['General Anxiety']}")
    print(f"  Individual Anxiety: {row['Individual Anxiety']}\n")

In [ ]:
# Calculate HRV metrics during HADS questions
ibi_hads_01 = get_ibi_data(questions_hads_01, ibi_01)
ibi_hads_02 = get_ibi_data(questions_hads_02, ibi_02)
ibi_hads_03 = get_ibi_data(questions_hads_03, ibi_03)

rmssd_hads_01, sdnn_hads_01 = calculate_hrv_metrics(ibi_hads_01)
rmssd_hads_02, sdnn_hads_02 = calculate_hrv_metrics(ibi_hads_02)
rmssd_hads_03, sdnn_hads_03 = calculate_hrv_metrics(ibi_hads_03)

# Calculate baseline from concatenated test IBIs
rmssd_bl_concat, sdnn_bl_concat = calculate_hrv_metrics(pd.concat([ibi_01['ibi'], ibi_02['ibi'], ibi_03['ibi']]))

# Check anxiety per test
anxiety_general_01, anxiety_individual_01 = determine_anxiety(rmssd_hads_01, sdnn_hads_01, rmssd_bl_concat, sdnn_bl_concat)
anxiety_general_02, anxiety_individual_02 = determine_anxiety(rmssd_hads_02, sdnn_hads_02, rmssd_bl_concat, sdnn_bl_concat)
anxiety_general_03, anxiety_individual_03 = determine_anxiety(rmssd_hads_03, sdnn_hads_03, rmssd_bl_concat, sdnn_bl_concat)

# Create results table
results_hads = pd.DataFrame({
    'Test': ['Test 01', 'Test 02', 'Test 03'],
    'RMSSD': [rmssd_hads_01, rmssd_hads_02, rmssd_hads_03],
    'SDNN': [sdnn_hads_01, sdnn_hads_02, sdnn_hads_03],
    'General Anxiety': ['Yes' if anxiety_general_01 else 'No', 'Yes' if anxiety_general_02 else 'No', 'Yes' if anxiety_general_03 else 'No'],
    'Individual Anxiety': ['Yes' if anxiety_individual_01 else 'No', 'Yes' if anxiety_individual_02 else 'No', 'Yes' if anxiety_individual_03 else 'No']
})

# Display results
print(f"Baseline RMSSD: {rmssd_bl_concat:.2f} ms, SDNN: {sdnn_bl_concat:.2f} ms\n")

for index, row in results_hads.iterrows():
    print(f"{row['Test']}:")
    print(f"  RMSSD during HADS: {row['RMSSD']:.2f} ms")
    print(f"  SDNN during HADS: {row['SDNN']:.2f} ms")
    print(f"  General Anxiety: {row['General Anxiety']}")
    print(f"  Individual Anxiety: {row['Individual Anxiety']}\n")

In [ ]:
# Calculate HRV metrics during HADS questions
ibi_hads_01 = get_ibi_data(questions_hads_01, ibi_01)
ibi_hads_02 = get_ibi_data(questions_hads_02, ibi_02)
ibi_hads_03 = get_ibi_data(questions_hads_03, ibi_03)

rmssd_hads_01, sdnn_hads_01 = calculate_hrv_metrics(ibi_hads_01)
rmssd_hads_02, sdnn_hads_02 = calculate_hrv_metrics(ibi_hads_02)
rmssd_hads_03, sdnn_hads_03 = calculate_hrv_metrics(ibi_hads_03)

# Calculate average HRV
average_rmssd = np.mean([rmssd_hads_01, rmssd_hads_02, rmssd_hads_03])
average_sdnn = np.mean([sdnn_hads_01, sdnn_hads_02, sdnn_hads_03])

# Calculate baseline from concatenated test IBIs
rmssd_bl_concat, sdnn_bl_concat = calculate_hrv_metrics(pd.concat([ibi_01['ibi'], ibi_02['ibi'], ibi_03['ibi']]))

# Check anxiety per test
anxiety_general_01, anxiety_individual_01 = determine_anxiety(rmssd_hads_01, sdnn_hads_01, rmssd_bl_concat, sdnn_bl_concat)
anxiety_general_02, anxiety_individual_02 = determine_anxiety(rmssd_hads_02, sdnn_hads_02, rmssd_bl_concat, sdnn_bl_concat)
anxiety_general_03, anxiety_individual_03 = determine_anxiety(rmssd_hads_03, sdnn_hads_03, rmssd_bl_concat, sdnn_bl_concat)

# Create results table
results_hads = pd.DataFrame({
    'Test': ['Test 01', 'Test 02', 'Test 03'],
    'RMSSD': [rmssd_hads_01, rmssd_hads_02, rmssd_hads_03],
    'SDNN': [sdnn_hads_01, sdnn_hads_02, sdnn_hads_03],
    'General Anxiety': ['Yes' if anxiety_general_01 else 'No', 'Yes' if anxiety_general_02 else 'No', 'Yes' if anxiety_general_03 else 'No'],
    'Individual Anxiety': ['Yes' if anxiety_individual_01 else 'No', 'Yes' if anxiety_individual_02 else 'No', 'Yes' if anxiety_individual_03 else 'No']
})

# Display HRV metrics
print(f"Average RMSSD: {average_rmssd:.2f} ms, Average SDNN: {average_sdnn:.2f} ms\n")

# Display results
for index, row in results_hads.iterrows():
    print(f"{row['Test']}:")
    print(f"  RMSSD during HADS: {row['RMSSD']:.2f} ms")
    print(f"  SDNN during HADS: {row['SDNN']:.2f} ms")
    print(f"  General Anxiety: {row['General Anxiety']}")
    print(f"  Individual Anxiety: {row['Individual Anxiety']}\n")

In [ ]:
# Calculate HRV metrics per test (using DataFrame-level calculate)
rmssd_bl, sdnn_bl = calculate_hrv_metrics(ibi_baseline['ibi'])
rmssd_01, sdnn_01 = calculate_hrv_metrics(ibi_01[ibi_01['ibi'] > 0]['ibi'])
rmssd_02, sdnn_02 = calculate_hrv_metrics(ibi_02[ibi_02['ibi'] > 0]['ibi'])
rmssd_03, sdnn_03 = calculate_hrv_metrics(ibi_03[ibi_03['ibi'] > 0]['ibi'])

# Check anxiety per test
anxiety_general_01, anxiety_individual_01 = determine_anxiety(rmssd_01, sdnn_01, rmssd_bl, sdnn_bl)
anxiety_general_02, anxiety_individual_02 = determine_anxiety(rmssd_02, sdnn_02, rmssd_bl, sdnn_bl)
anxiety_general_03, anxiety_individual_03 = determine_anxiety(rmssd_03, sdnn_03, rmssd_bl, sdnn_bl)

# Display the results
print(f"Baseline RMSSD: {rmssd_bl:.2f} ms, SDNN: {sdnn_bl:.2f} ms")
print(f"Test 01 RMSSD: {rmssd_01:.2f} ms, SDNN: {sdnn_01:.2f} ms, General Anxiety: {'Yes' if anxiety_general_01 else 'No'}, Individual Anxiety: {'Yes' if anxiety_individual_01 else 'No'}")
print(f"Test 02 RMSSD: {rmssd_02:.2f} ms, SDNN: {sdnn_02:.2f} ms, General Anxiety: {'Yes' if anxiety_general_02 else 'No'}, Individual Anxiety: {'Yes' if anxiety_individual_02 else 'No'}")
print(f"Test 03 RMSSD: {rmssd_03:.2f} ms, SDNN: {sdnn_03:.2f} ms, General Anxiety: {'Yes' if anxiety_general_03 else 'No'}, Individual Anxiety: {'Yes' if anxiety_individual_03 else 'No'}")

In [ ]:
# Calculate HRV metrics during HADS questions
ibi_hads_01 = get_ibi_data(questions_hads_01, ibi_01)
ibi_hads_02 = get_ibi_data(questions_hads_02, ibi_02)
ibi_hads_03 = get_ibi_data(questions_hads_03, ibi_03)

rmssd_hads_01, sdnn_hads_01 = calculate_hrv_metrics(ibi_hads_01)
rmssd_hads_02, sdnn_hads_02 = calculate_hrv_metrics(ibi_hads_02)
rmssd_hads_03, sdnn_hads_03 = calculate_hrv_metrics(ibi_hads_03)

# Calculate average baseline
ibi_bl_concat = pd.concat([ibi_01['ibi'], ibi_02['ibi'], ibi_03['ibi']])
rmssd_bl_concat, sdnn_bl_concat = calculate_hrv_metrics(ibi_bl_concat)

# Check anxiety per test
anxiety_general_01, anxiety_individual_01 = determine_anxiety(rmssd_hads_01, sdnn_hads_01, rmssd_bl_concat, sdnn_bl_concat)
anxiety_general_02, anxiety_individual_02 = determine_anxiety(rmssd_hads_02, sdnn_hads_02, rmssd_bl_concat, sdnn_bl_concat)
anxiety_general_03, anxiety_individual_03 = determine_anxiety(rmssd_hads_03, sdnn_hads_03, rmssd_bl_concat, sdnn_bl_concat)

# Create results table
results_hads = pd.DataFrame({
    'Test': ['Test 01', 'Test 02', 'Test 03'],
    'RMSSD': [rmssd_hads_01, rmssd_hads_02, rmssd_hads_03],
    'SDNN': [sdnn_hads_01, sdnn_hads_02, sdnn_hads_03],
    'General Anxiety': ['Yes' if anxiety_general_01 else 'No', 'Yes' if anxiety_general_02 else 'No', 'Yes' if anxiety_general_03 else 'No'],
    'Individual Anxiety': ['Yes' if anxiety_individual_01 else 'No', 'Yes' if anxiety_individual_02 else 'No', 'Yes' if anxiety_individual_03 else 'No']
})

# Display results
print(f"Baseline RMSSD: {rmssd_bl_concat:.2f} ms, SDNN: {sdnn_bl_concat:.2f} ms\n")

for index, row in results_hads.iterrows():
    print(f"{row['Test']}:")
    print(f"  RMSSD during HADS: {row['RMSSD']:.2f} ms")
    print(f"  SDNN during HADS: {row['SDNN']:.2f} ms")
    print(f"  General Anxiety: {row['General Anxiety']}")
    print(f"  Individual Anxiety: {row['Individual Anxiety']}\n")

In [ ]:
# Calculate HRV metrics during HADS questions (with valid IBI filtering)
ibi_hads_01 = get_ibi_data(questions_hads_01, ibi_01)
ibi_hads_02 = get_ibi_data(questions_hads_02, ibi_02)
ibi_hads_03 = get_ibi_data(questions_hads_03, ibi_03)

# Ensure valid IBI
ibi_hads_01 = ibi_hads_01[ibi_hads_01 > 0]
ibi_hads_02 = ibi_hads_02[ibi_hads_02 > 0]
ibi_hads_03 = ibi_hads_03[ibi_hads_03 > 0]

# Calculate HRV metrics
rmssd_hads_01, sdnn_hads_01 = calculate_hrv_metrics(ibi_hads_01)
rmssd_hads_02, sdnn_hads_02 = calculate_hrv_metrics(ibi_hads_02)
rmssd_hads_03, sdnn_hads_03 = calculate_hrv_metrics(ibi_hads_03)

# Check anxiety per test
anxiety_general_01, anxiety_individual_01 = determine_anxiety(rmssd_hads_01, sdnn_hads_01, rmssd_baseline, sdnn_baseline)
anxiety_general_02, anxiety_individual_02 = determine_anxiety(rmssd_hads_02, sdnn_hads_02, rmssd_baseline, sdnn_baseline)
anxiety_general_03, anxiety_individual_03 = determine_anxiety(rmssd_hads_03, sdnn_hads_03, rmssd_baseline, sdnn_baseline)

# Create results table
results_hads = pd.DataFrame({
    'Test': ['Test 01', 'Test 02', 'Test 03'],
    'RMSSD': [rmssd_hads_01, rmssd_hads_02, rmssd_hads_03],
    'SDNN': [sdnn_hads_01, sdnn_hads_02, sdnn_hads_03],
    'General Anxiety': ['Yes' if anxiety_general_01 else 'No', 'Yes' if anxiety_general_02 else 'No', 'Yes' if anxiety_general_03 else 'No'],
    'Individual Anxiety': ['Yes' if anxiety_individual_01 else 'No', 'Yes' if anxiety_individual_02 else 'No', 'Yes' if anxiety_individual_03 else 'No']
})

# Display HRV metrics
print(f"Baseline RMSSD: {rmssd_baseline:.2f} ms, SDNN: {sdnn_baseline:.2f} ms\n")

# Display results
for index, row in results_hads.iterrows():
    print(f"{row['Test']}:")
    print(f"  RMSSD during HADS: {row['RMSSD']:.2f} ms")
    print(f"  SDNN during HADS: {row['SDNN']:.2f} ms")
    print(f"  General Anxiety: {row['General Anxiety']}")
    print(f"  Individual Anxiety: {row['Individual Anxiety']}\n")

In [ ]:
# Calculate HRV metrics during HADS questions (with valid IBI filtering)
ibi_hads_01 = get_ibi_data(questions_hads_01, ibi_01)
ibi_hads_02 = get_ibi_data(questions_hads_02, ibi_02)
ibi_hads_03 = get_ibi_data(questions_hads_03, ibi_03)

# Ensure valid IBI
ibi_hads_01 = ibi_hads_01[ibi_hads_01 > 0]
ibi_hads_02 = ibi_hads_02[ibi_hads_02 > 0]
ibi_hads_03 = ibi_hads_03[ibi_hads_03 > 0]

# Calculate HRV metrics
rmssd_hads_01, sdnn_hads_01 = calculate_hrv_metrics(ibi_hads_01)
rmssd_hads_02, sdnn_hads_02 = calculate_hrv_metrics(ibi_hads_02)
rmssd_hads_03, sdnn_hads_03 = calculate_hrv_metrics(ibi_hads_03)

# Check anxiety per test
anxiety_general_01, anxiety_individual_01 = determine_anxiety(rmssd_hads_01, sdnn_hads_01, rmssd_baseline, sdnn_baseline)
anxiety_general_02, anxiety_individual_02 = determine_anxiety(rmssd_hads_02, sdnn_hads_02, rmssd_baseline, sdnn_baseline)
anxiety_general_03, anxiety_individual_03 = determine_anxiety(rmssd_hads_03, sdnn_hads_03, rmssd_baseline, sdnn_baseline)

# Create results table
results_hads = pd.DataFrame({
    'Test': ['Test 01', 'Test 02', 'Test 03'],
    'Start Time': [
        questions_hads_01['Question Start Time'].min().strftime('%H:%M:%S'),
        questions_hads_02['Question Start Time'].min().strftime('%H:%M:%S'),
        questions_hads_03['Question Start Time'].min().strftime('%H:%M:%S')
    ],
    'End Time': [
        questions_hads_01['Question Answer Time'].max().strftime('%H:%M:%S'),
        questions_hads_02['Question Answer Time'].max().strftime('%H:%M:%S'),
        questions_hads_03['Question Answer Time'].max().strftime('%H:%M:%S')
    ],
    'RMSSD': [rmssd_hads_01, rmssd_hads_02, rmssd_hads_03],
    'SDNN': [sdnn_hads_01, sdnn_hads_02, sdnn_hads_03],
    'General Anxiety (RMSSD)': ['Yes' if anxiety_general_01 else 'No', 'Yes' if anxiety_general_02 else 'No', 'Yes' if anxiety_general_03 else 'No'],
    'General Anxiety (SDNN)': ['Yes' if sdnn_hads_01 < sdnn_threshold else 'No', 'Yes' if sdnn_hads_02 < sdnn_threshold else 'No', 'Yes' if sdnn_hads_03 < sdnn_threshold else 'No'],
    'Individual Anxiety (RMSSD)': ['Yes' if anxiety_individual_01 else 'No', 'Yes' if anxiety_individual_02 else 'No', 'Yes' if anxiety_individual_03 else 'No'],
    'Individual Anxiety (SDNN)': ['Yes' if sdnn_hads_01 < sdnn_baseline else 'No', 'Yes' if sdnn_hads_02 < sdnn_baseline else 'No', 'Yes' if sdnn_hads_03 < sdnn_baseline else 'No']
})

# Display HRV metrics
print(f"Baseline RMSSD: {rmssd_baseline:.2f} ms, SDNN: {sdnn_baseline:.2f} ms\n")

# Display results
for index, row in results_hads.iterrows():
    print(f"{row['Test']}:")
    print(f"  Start time for HADS: {row['Start Time']}")
    print(f"  End time for HADS: {row['End Time']}")
    print(f"  RMSSD during HADS: {row['RMSSD']:.2f} ms")
    print(f"  SDNN during HADS: {row['SDNN']:.2f} ms")
    print(f"  General Anxiety (RMSSD): {row['General Anxiety (RMSSD)']}")
    print(f"  General Anxiety (SDNN): {row['General Anxiety (SDNN)']}")
    print(f"  Individual Anxiety (RMSSD): {row['Individual Anxiety (RMSSD)']}")
    print(f"  Individual Anxiety (SDNN): {row['Individual Anxiety (SDNN)']}\n")

In [ ]:
# Calculate HRV metrics during STAI-S questions
ibi_stais_01 = get_ibi_data(questions_stais_01, ibi_01)
ibi_stais_02 = get_ibi_data(questions_stais_02, ibi_02)
ibi_stais_03 = get_ibi_data(questions_stais_03, ibi_03)

# Ensure valid IBI
ibi_stais_01 = ibi_stais_01[ibi_stais_01 > 0]
ibi_stais_02 = ibi_stais_02[ibi_stais_02 > 0]
ibi_stais_03 = ibi_stais_03[ibi_stais_03 > 0]

# Calculate HRV metrics
rmssd_stais_01, sdnn_stais_01 = calculate_hrv_metrics(ibi_stais_01)
rmssd_stais_02, sdnn_stais_02 = calculate_hrv_metrics(ibi_stais_02)
rmssd_stais_03, sdnn_stais_03 = calculate_hrv_metrics(ibi_stais_03)

# Check anxiety per test
anxiety_general_01, anxiety_individual_01 = determine_anxiety(rmssd_stais_01, sdnn_stais_01, rmssd_baseline, sdnn_baseline)
anxiety_general_02, anxiety_individual_02 = determine_anxiety(rmssd_stais_02, sdnn_stais_02, rmssd_baseline, sdnn_baseline)
anxiety_general_03, anxiety_individual_03 = determine_anxiety(rmssd_stais_03, sdnn_stais_03, rmssd_baseline, sdnn_baseline)

# Create results table
results_stais = pd.DataFrame({
    'Test': ['Test 01', 'Test 02', 'Test 03'],
    'Start Time': [
        questions_stais_01['Question Start Time'].min().strftime('%H:%M:%S'),
        questions_stais_02['Question Start Time'].min().strftime('%H:%M:%S'),
        questions_stais_03['Question Start Time'].min().strftime('%H:%M:%S')
    ],
    'End Time': [
        questions_stais_01['Question Answer Time'].max().strftime('%H:%M:%S'),
        questions_stais_02['Question Answer Time'].max().strftime('%H:%M:%S'),
        questions_stais_03['Question Answer Time'].max().strftime('%H:%M:%S')
    ],
    'RMSSD': [rmssd_stais_01, rmssd_stais_02, rmssd_stais_03],
    'SDNN': [sdnn_stais_01, sdnn_stais_02, sdnn_stais_03],
    'General Anxiety (RMSSD)': ['Yes' if anxiety_general_01 else 'No', 'Yes' if anxiety_general_02 else 'No', 'Yes' if anxiety_general_03 else 'No'],
    'General Anxiety (SDNN)': ['Yes' if sdnn_stais_01 < sdnn_threshold else 'No', 'Yes' if sdnn_stais_02 < sdnn_threshold else 'No', 'Yes' if sdnn_stais_03 < sdnn_threshold else 'No'],
    'Individual Anxiety (RMSSD)': ['Yes' if anxiety_individual_01 else 'No', 'Yes' if anxiety_individual_02 else 'No', 'Yes' if anxiety_individual_03 else 'No'],
    'Individual Anxiety (SDNN)': ['Yes' if sdnn_stais_01 < sdnn_baseline else 'No', 'Yes' if sdnn_stais_02 < sdnn_baseline else 'No', 'Yes' if sdnn_stais_03 < sdnn_baseline else 'No']
})

# Display HRV metrics
print(f"Baseline RMSSD: {rmssd_baseline:.2f} ms, SDNN: {sdnn_baseline:.2f} ms\n")

# Display results
for index, row in results_stais.iterrows():
    print(f"{row['Test']}:")
    print(f"  Start time for STAI-S: {row['Start Time']}")
    print(f"  End time for STAI-S: {row['End Time']}")
    print(f"  RMSSD during STAI-S: {row['RMSSD']:.2f} ms")
    print(f"  SDNN during STAI-S: {row['SDNN']:.2f} ms")
    print(f"  General Anxiety (RMSSD): {row['General Anxiety (RMSSD)']}")
    print(f"  General Anxiety (SDNN): {row['General Anxiety (SDNN)']}")
    print(f"  Individual Anxiety (RMSSD): {row['Individual Anxiety (RMSSD)']}")
    print(f"  Individual Anxiety (SDNN): {row['Individual Anxiety (SDNN)']}\n")

In [ ]:
# Calculate HRV metrics during STAI-T questions
ibi_stait_01 = get_ibi_data(questions_stait_01, ibi_01)
ibi_stait_02 = get_ibi_data(questions_stait_02, ibi_02)
ibi_stait_03 = get_ibi_data(questions_stait_03, ibi_03)

# Ensure valid IBI
ibi_stait_01 = ibi_stait_01[ibi_stait_01 > 0]
ibi_stait_02 = ibi_stait_02[ibi_stait_02 > 0]
ibi_stait_03 = ibi_stait_03[ibi_stait_03 > 0]

# Calculate HRV metrics
rmssd_stait_01, sdnn_stait_01 = calculate_hrv_metrics(ibi_stait_01)
rmssd_stait_02, sdnn_stait_02 = calculate_hrv_metrics(ibi_stait_02)
rmssd_stait_03, sdnn_stait_03 = calculate_hrv_metrics(ibi_stait_03)

# Check anxiety per test
anxiety_general_01, anxiety_individual_01 = determine_anxiety(rmssd_stait_01, sdnn_stait_01, rmssd_baseline, sdnn_baseline)
anxiety_general_02, anxiety_individual_02 = determine_anxiety(rmssd_stait_02, sdnn_stait_02, rmssd_baseline, sdnn_baseline)
anxiety_general_03, anxiety_individual_03 = determine_anxiety(rmssd_stait_03, sdnn_stait_03, rmssd_baseline, sdnn_baseline)

# Create results table
results_stait = pd.DataFrame({
    'Test': ['Test 01', 'Test 02', 'Test 03'],
    'Start Time': [
        questions_stait_01['Question Start Time'].min().strftime('%H:%M:%S'),
        questions_stait_02['Question Start Time'].min().strftime('%H:%M:%S'),
        questions_stait_03['Question Start Time'].min().strftime('%H:%M:%S')
    ],
    'End Time': [
        questions_stait_01['Question Answer Time'].max().strftime('%H:%M:%S'),
        questions_stait_02['Question Answer Time'].max().strftime('%H:%M:%S'),
        questions_stait_03['Question Answer Time'].max().strftime('%H:%M:%S')
    ],
    'RMSSD': [rmssd_stait_01, rmssd_stait_02, rmssd_stait_03],
    'SDNN': [sdnn_stait_01, sdnn_stait_02, sdnn_stait_03],
    'General Anxiety (RMSSD)': ['Yes' if anxiety_general_01 else 'No', 'Yes' if anxiety_general_02 else 'No', 'Yes' if anxiety_general_03 else 'No'],
    'General Anxiety (SDNN)': ['Yes' if sdnn_stait_01 < sdnn_threshold else 'No', 'Yes' if sdnn_stait_02 < sdnn_threshold else 'No', 'Yes' if sdnn_stait_03 < sdnn_threshold else 'No'],
    'Individual Anxiety (RMSSD)': ['Yes' if anxiety_individual_01 else 'No', 'Yes' if anxiety_individual_02 else 'No', 'Yes' if anxiety_individual_03 else 'No'],
    'Individual Anxiety (SDNN)': ['Yes' if sdnn_stait_01 < sdnn_baseline else 'No', 'Yes' if sdnn_stait_02 < sdnn_baseline else 'No', 'Yes' if sdnn_stait_03 < sdnn_baseline else 'No']
})

# Display HRV metrics
print(f"Baseline RMSSD: {rmssd_baseline:.2f} ms, SDNN: {sdnn_baseline:.2f} ms\n")

# Display results
for index, row in results_stait.iterrows():
    print(f"{row['Test']}:")
    print(f"  Start time for STAI-T: {row['Start Time']}")
    print(f"  End time for STAI-T: {row['End Time']}")
    print(f"  RMSSD during STAI-T: {row['RMSSD']:.2f} ms")
    print(f"  SDNN during STAI-T: {row['SDNN']:.2f} ms")
    print(f"  General Anxiety (RMSSD): {row['General Anxiety (RMSSD)']}")
    print(f"  General Anxiety (SDNN): {row['General Anxiety (SDNN)']}")
    print(f"  Individual Anxiety (RMSSD): {row['Individual Anxiety (RMSSD)']}")
    print(f"  Individual Anxiety (SDNN): {row['Individual Anxiety (SDNN)']}\n")

In [ ]:
# Calculate HRV metrics during BFI questions
ibi_bfi_01 = get_ibi_data(questions_bfi_01, ibi_01)
ibi_bfi_02 = get_ibi_data(questions_bfi_02, ibi_02)
ibi_bfi_03 = get_ibi_data(questions_bfi_03, ibi_03)

# Ensure valid IBI
ibi_bfi_01 = ibi_bfi_01[ibi_bfi_01 > 0]
ibi_bfi_02 = ibi_bfi_02[ibi_bfi_02 > 0]
ibi_bfi_03 = ibi_bfi_03[ibi_bfi_03 > 0]

# Calculate HRV metrics
rmssd_bfi_01, sdnn_bfi_01 = calculate_hrv_metrics(ibi_bfi_01)
rmssd_bfi_02, sdnn_bfi_02 = calculate_hrv_metrics(ibi_bfi_02)
rmssd_bfi_03, sdnn_bfi_03 = calculate_hrv_metrics(ibi_bfi_03)

# Check anxiety per test
anxiety_general_01, anxiety_individual_01 = determine_anxiety(rmssd_bfi_01, sdnn_bfi_01, rmssd_baseline, sdnn_baseline)
anxiety_general_02, anxiety_individual_02 = determine_anxiety(rmssd_bfi_02, sdnn_bfi_02, rmssd_baseline, sdnn_baseline)
anxiety_general_03, anxiety_individual_03 = determine_anxiety(rmssd_bfi_03, sdnn_bfi_03, rmssd_baseline, sdnn_baseline)

# Create results table
results_bfi = pd.DataFrame({
    'Test': ['Test 01', 'Test 02', 'Test 03'],
    'Start Time': [
        questions_bfi_01['Question Start Time'].min().strftime('%H:%M:%S'),
        questions_bfi_02['Question Start Time'].min().strftime('%H:%M:%S'),
        questions_bfi_03['Question Start Time'].min().strftime('%H:%M:%S')
    ],
    'End Time': [
        questions_bfi_01['Question Answer Time'].max().strftime('%H:%M:%S'),
        questions_bfi_02['Question Answer Time'].max().strftime('%H:%M:%S'),
        questions_bfi_03['Question Answer Time'].max().strftime('%H:%M:%S')
    ],
    'RMSSD': [rmssd_bfi_01, rmssd_bfi_02, rmssd_bfi_03],
    'SDNN': [sdnn_bfi_01, sdnn_bfi_02, sdnn_bfi_03],
    'General Anxiety (RMSSD)': ['Yes' if anxiety_general_01 else 'No', 'Yes' if anxiety_general_02 else 'No', 'Yes' if anxiety_general_03 else 'No'],
    'General Anxiety (SDNN)': ['Yes' if sdnn_bfi_01 < sdnn_threshold else 'No', 'Yes' if sdnn_bfi_02 < sdnn_threshold else 'No', 'Yes' if sdnn_bfi_03 < sdnn_threshold else 'No'],
    'Individual Anxiety (RMSSD)': ['Yes' if anxiety_individual_01 else 'No', 'Yes' if anxiety_individual_02 else 'No', 'Yes' if anxiety_individual_03 else 'No'],
    'Individual Anxiety (SDNN)': ['Yes' if sdnn_bfi_01 < sdnn_baseline else 'No', 'Yes' if sdnn_bfi_02 < sdnn_baseline else 'No', 'Yes' if sdnn_bfi_03 < sdnn_baseline else 'No']
})

# Display HRV metrics
print(f"Baseline RMSSD: {rmssd_baseline:.2f} ms, SDNN: {sdnn_baseline:.2f} ms\n")

# Display results
for index, row in results_bfi.iterrows():
    print(f"{row['Test']}:")
    print(f"  Start time for BFI: {row['Start Time']}")
    print(f"  End time for BFI: {row['End Time']}")
    print(f"  RMSSD during BFI: {row['RMSSD']:.2f} ms")
    print(f"  SDNN during BFI: {row['SDNN']:.2f} ms")
    print(f"  General Anxiety (RMSSD): {row['General Anxiety (RMSSD)']}")
    print(f"  General Anxiety (SDNN): {row['General Anxiety (SDNN)']}")
    print(f"  Individual Anxiety (RMSSD): {row['Individual Anxiety (RMSSD)']}")
    print(f"  Individual Anxiety (SDNN): {row['Individual Anxiety (SDNN)']}\n")

In [ ]:
# Calculate HRV metrics during FQ questions
ibi_fq_01 = get_ibi_data(questions_fq_01, ibi_01)
ibi_fq_02 = get_ibi_data(questions_fq_02, ibi_02)
ibi_fq_03 = get_ibi_data(questions_fq_03, ibi_03)

# Ensure valid IBI
ibi_fq_01 = ibi_fq_01[ibi_fq_01 > 0]
ibi_fq_02 = ibi_fq_02[ibi_fq_02 > 0]
ibi_fq_03 = ibi_fq_03[ibi_fq_03 > 0]

# Calculate HRV metrics
rmssd_fq_01, sdnn_fq_01 = calculate_hrv_metrics(ibi_fq_01)
rmssd_fq_02, sdnn_fq_02 = calculate_hrv_metrics(ibi_fq_02)
rmssd_fq_03, sdnn_fq_03 = calculate_hrv_metrics(ibi_fq_03)

# Check anxiety per test
anxiety_general_01, anxiety_individual_01 = determine_anxiety(rmssd_fq_01, sdnn_fq_01, rmssd_baseline, sdnn_baseline)
anxiety_general_02, anxiety_individual_02 = determine_anxiety(rmssd_fq_02, sdnn_fq_02, rmssd_baseline, sdnn_baseline)
anxiety_general_03, anxiety_individual_03 = determine_anxiety(rmssd_fq_03, sdnn_fq_03, rmssd_baseline, sdnn_baseline)

# Create results table
results_fq = pd.DataFrame({
    'Test': ['Test 01', 'Test 02', 'Test 03'],
    'Start Time': [
        questions_fq_01['Question Start Time'].min().strftime('%H:%M:%S'),
        questions_fq_02['Question Start Time'].min().strftime('%H:%M:%S'),
        questions_fq_03['Question Start Time'].min().strftime('%H:%M:%S')
    ],
    'End Time': [
        questions_fq_01['Question Answer Time'].max().strftime('%H:%M:%S'),
        questions_fq_02['Question Answer Time'].max().strftime('%H:%M:%S'),
        questions_fq_03['Question Answer Time'].max().strftime('%H:%M:%S')
    ],
    'RMSSD': [rmssd_fq_01, rmssd_fq_02, rmssd_fq_03],
    'SDNN': [sdnn_fq_01, sdnn_fq_02, sdnn_fq_03],
    'General Anxiety (RMSSD)': ['Yes' if anxiety_general_01 else 'No', 'Yes' if anxiety_general_02 else 'No', 'Yes' if anxiety_general_03 else 'No'],
    'General Anxiety (SDNN)': ['Yes' if sdnn_fq_01 < sdnn_threshold else 'No', 'Yes' if sdnn_fq_02 < sdnn_threshold else 'No', 'Yes' if sdnn_fq_03 < sdnn_threshold else 'No'],
    'Individual Anxiety (RMSSD)': ['Yes' if anxiety_individual_01 else 'No', 'Yes' if anxiety_individual_02 else 'No', 'Yes' if anxiety_individual_03 else 'No'],
    'Individual Anxiety (SDNN)': ['Yes' if sdnn_fq_01 < sdnn_baseline else 'No', 'Yes' if sdnn_fq_02 < sdnn_baseline else 'No', 'Yes' if sdnn_fq_03 < sdnn_baseline else 'No']
})

# Display HRV metrics
print(f"Baseline RMSSD: {rmssd_baseline:.2f} ms, SDNN: {sdnn_baseline:.2f} ms\n")

# Display results
for index, row in results_fq.iterrows():
    print(f"{row['Test']}:")
    print(f"  Start time for FQ: {row['Start Time']}")
    print(f"  End time for FQ: {row['End Time']}")
    print(f"  RMSSD during FQ: {row['RMSSD']:.2f} ms")
    print(f"  SDNN during FQ: {row['SDNN']:.2f} ms")
    print(f"  General Anxiety (RMSSD): {row['General Anxiety (RMSSD)']}")
    print(f"  General Anxiety (SDNN): {row['General Anxiety (SDNN)']}")
    print(f"  Individual Anxiety (RMSSD): {row['Individual Anxiety (RMSSD)']}")
    print(f"  Individual Anxiety (SDNN): {row['Individual Anxiety (SDNN)']}\n")

In [ ]:
# Process all question types using process_questions helper
for question_type in question_types:
    results = process_questions(question_type, psychometric_data, ibi_data, rmssd_baseline, sdnn_baseline)
    print(f"\nResults for {question_type} questions:")
    print(f"Baseline RMSSD: {rmssd_baseline:.2f} ms, SDNN: {sdnn_baseline:.2f} ms\n")
    for index, row in results.iterrows():
        print(f"{row['Test']}:")
        print(f"  Start time for {question_type}: {row['Start Time']}")
        print(f"  End time for {question_type}: {row['End Time']}")
        print(f"  RMSSD during {question_type}: {row['RMSSD']:.2f} ms")
        print(f"  SDNN during {question_type}: {row['SDNN']:.2f} ms")
        print(f"  General Anxiety (RMSSD): {row['General Anxiety (RMSSD)']}")
        print(f"  General Anxiety (SDNN): {row['General Anxiety (SDNN)']}")
        print(f"  Individual Anxiety (RMSSD): {row['Individual Anxiety (RMSSD)']}")
        print(f"  Individual Anxiety (SDNN): {row['Individual Anxiety (SDNN)']}\n")

In [ ]:
# Process all question types and collect into one DataFrame
all_results = pd.DataFrame()

for question_type in question_types:
    results = process_questions(question_type, psychometric_data, ibi_data, rmssd_baseline, sdnn_baseline)
    results.insert(1, 'Type', question_type)
    all_results = pd.concat([all_results, results])

# Display HRV metrics
print(f"Baseline RMSSD: {rmssd_baseline:.2f} ms, SDNN: {sdnn_baseline:.2f} ms\n")

# Display results
for index, row in all_results.iterrows():
    print(f"{row['Test']}:")
    print(f"  Start time for {row['Type']}: {row['Start Time']}")
    print(f"  End time for {row['Type']}: {row['End Time']}")
    print(f"  RMSSD during {row['Type']}: {row['RMSSD']:.2f} ms")
    print(f"  SDNN during {row['Type']}: {row['SDNN']:.2f} ms")
    print(f"  General Anxiety (RMSSD): {row['General Anxiety (RMSSD)']}")
    print(f"  General Anxiety (SDNN): {row['General Anxiety (SDNN)']}")
    print(f"  Individual Anxiety (RMSSD): {row['Individual Anxiety (RMSSD)']}")
    print(f"  Individual Anxiety (SDNN): {row['Individual Anxiety (SDNN)']}\n")

In [ ]:
# Process all question types (simplified results with RMSSD/SDNN only)
all_results = pd.DataFrame()

for question_type in question_types:
    results = process_questions(question_type, psychometric_data, ibi_data, rmssd_baseline, sdnn_baseline)
    results.insert(1, 'Type', question_type)
    all_results = pd.concat([all_results, results])

# Display HRV metrics
print(f"Baseline RMSSD: {rmssd_baseline:.2f} ms, SDNN: {sdnn_baseline:.2f} ms\n")

# Show results
all_results = all_results.reset_index(drop=True)
print(all_results.to_string(index=False))